In [8]:
import jupyter_black

jupyter_black.load()

In [9]:
# Importing Libraries

import numpy as np
import astropy
import photutils
import ccdproc
from ccdproc import CCDData, combiner
from astropy import units as u
import astropy.io.fits as fits
from astropy.io import ascii

from astropy.visualization import SqrtStretch
from astropy.visualization.mpl_normalize import ImageNormalize
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from photutils.centroids import centroid_com, centroid_1dg, centroid_2dg
from photutils.aperture import CircularAperture
from photutils.aperture import aperture_photometry
from photutils.background import Background2D
from photutils.background import MedianBackground
from photutils.detection import DAOStarFinder
from photutils.segmentation import detect_sources, deblend_sources, SourceCatalog
from scipy.ndimage import shift
import astroalign as aa
import gc

from astropy.coordinates import SkyCoord

In [10]:
# Path for scaled images
path = "Aligned_S50_faeae72a_20260321_debayer/NGC_2175_sub/"
r_images = ccdproc.ImageFileCollection(path, glob_include="*r.fit")
g_images = ccdproc.ImageFileCollection(path, glob_include="*g.fit")
b_images = ccdproc.ImageFileCollection(path, glob_include="*b.fit")

all_names = b_images.files + g_images.files + r_images.files

# Target path for scaled images
apath = "ScaledAligned_S50_faeae72a_20260321_debayer/NGC_2175_sub/"

In [11]:
positions = [(537, 951), (880, 1061), (642, 1130), (618, 1537)]
apertures = CircularAperture(positions, r=10.0)

In [17]:
refidx = 40  # Reference image
# Need reference image to persist in memory
ref_filename = all_names[refidx]
ref_ccd = CCDData.read(path + ref_filename, unit="adu")
img_ref = ref_ccd.data + 0
ref_wcs = WCS(ref_ccd.header)

ref_apertures = aperture_photometry(img_ref, apertures)

failed_images = 0

print(f"Starting loop for {len(all_names)} images...")

for idx, filename in enumerate(all_names):

    if idx % np.floor(len(all_names) / 20) == 0:
        print(f"[-] Status: {idx}/{len(all_names)}")

    # Define images to be scaled
    current_ccd = CCDData.read(path + filename)
    # img_example = current_ccd.data.astype(np.float32, copy=False)

    try:
        img_apertures = aperture_photometry(current_ccd, apertures)
        normal_difference = (
            ref_apertures["aperture_sum"] / img_apertures["aperture_sum"]
        )
        median_scaling = np.ma.median(normal_difference)

        img_scaled = current_ccd.multiply(median_scaling)

        scaled_ccd = img_scaled
        scaled_ccd.header = current_ccd.header.copy()
        scaled_ccd.header["HISTORY"] = f"Scaled {filename} to {ref_filename}"

        scaled_ccd.write(
            apath + "scaled" + all_names[idx], overwrite=True
        )  # Saving images as files

    except Exception as e:
        failed_images += 1
        print(f"Image {idx} failed ({failed_images}): {e}")


print(f"Successfully scaled {len(all_names)-failed_images}/{len(all_names)} images.")

INFO:astropy:using the unit adu passed to the FITS reader instead of the unit adu in the FITS file.


INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
Starting loop for 870 images...
[-] Status: 0/870
[-] Status: 43/870
[-] Status: 86/870
[-] Status: 129/870
[-] Status: 172/870
[-] Status: 215/870
[-] Status: 258/870
[-] Status: 301/870
[-] Status: 344/870
[-] Status: 387/870
[-] Status: 430/870
[-] Status: 473/870
[-] Status: 516/870
[-] Status: 559/870
[-] Status: 602/870
[-] Status: 645/870
[-] Status: 688/870
[-] Status: 731/870
[-] Status: 774/870
[-] Status: 817/870
[-] Status: 860/870
Successfully scaled 870/870 images.
